# Set orientation

Measure how the camera image is rotated relative to the stage, so the
driver can align every saved image to the stage axes at save time.

Setup order for a new scope: **`set_stage_limits` -> `set_orientation` ->
`calibrate_objective_pair`**. Run this once per rig (re-run only if the
optical path changes). It is a separate concern from pixel-scale
calibration and from the stage limits.


## Step 1: Configure

Fill in the cell below:

- `session_id`: a name for this run (the date works well).
- `reference_objective`: which objective is on the microscope.
- `stage_move_um`: how far the stage should move during the test, in micrometers.

`SESSIONS_ROOT` is where this run's images and reports are saved. Edit that
line if you want a different location. The measured orientation is adopted
separately in Step 3 into `orientation/current.json`.

Running this cell opens LAS X and prepares the session folder. Nothing is
acquired yet.


In [ ]:
import _bootstrap
from pathlib import Path
from navigator_expert.orientation import measure as wf

SESSIONS_ROOT = Path(
    r"C:\ProgramData\MinicondaZMB\home\t.de\navigator_expert_calibration\sessions"
)

session = wf.start_session(
    session_id="2026-05-22_scope_orientation",
    job_name="Overview",
    reference_objective="10x",
    stage_move_um=40.0,
    sessions_root=SESSIONS_ROOT,
)
print(session)


## Step 2: Measure

The microscope takes three pictures: one at the home position, one after
moving +X, one after moving +Y. The notebook registers them, fits the 2x2
Jacobian, snaps it to the nearest 90-degree rotation, and rejects a
reflection or a poor fit (both mean a physical alignment problem, not
something to resample).

It prints the measured orientation and the D4 residual. A staging
`orientation.json` is written only when the fit is accepted.


In [ ]:
session = wf.measure(session)

if session.config_written:
    print(f"Orientation: rotate_deg={session.orientation.rotate_deg}, "
          f"mirror={session.orientation.mirror}  "
          f"(D4 residual {session.residual_from_d4:.3f})")
else:
    print(f"No orientation written. Reason: {session.failure_reason}")


## Step 3 (optional): Make this the active orientation

Run this cell only if Step 2 accepted the fit. It writes the measured
value to `orientation/current.json`, which the driver reads at save time to
reorient every exported plane to the stage axes. Until this runs, the rig
uses the shipped identity template (no reorientation).


In [ ]:
wf.adopt_orientation(session)
